# Milestone 03 - Leakage-Safe Preprocessing & PyTorch Data Pipeline

## Project
Bank Account Fraud Detection - Deep Learning & MLOps

## Purpose

This notebook will design and validate a preprocessing pipeline that can be reused by both classical baseline models and the later PyTorch neural network.

The central rule is that all learned preprocessing must be fit using the training partition only.

## Locked Split

The chronological split from Milestone 02 is now fixed:

- Months 0–5 → Train
- Month 6 → Validation
- Month 7 → Sealed Final Test

The final test set must not influence preprocessing, feature engineering, model selection, calibration, or threshold decisions.

## Initial Feature Policy

### Excluded from model inputs

- `fraud_bool` — target label
- `month` — used for splitting, temporal analysis, and monitoring
- `device_fraud_count` — constant at zero in `Base.csv`

### Categorical features

The following string features will be treated as categorical:

- `payment_type`
- `employment_status`
- `housing_status`
- `source`
- `device_os`

Categorical encoders must be fit on training data only.

Unknown categories appearing in validation or test data must not crash the pipeline.

### Binary indicators

The following features will remain numeric 0/1 indicators:

- `email_is_free`
- `phone_home_valid`
- `phone_mobile_valid`
- `has_other_cards`
- `foreign_request`
- `keep_alive_session`

These will not require one-hot encoding.

### Sentinel-coded numeric features

The following features contain possible `-1` sentinel states:

- `prev_address_months_count`
- `current_address_months_count`
- `bank_months_count`
- `session_length_in_minutes`
- `device_distinct_emails_8w`

The `-1` values will not be blindly replaced with medians.

Because Milestone 01 showed that some sentinel states have different fraud rates from ordinary values, the initial strategy will preserve the original numeric value and add an explicit indicator showing whether the value equals `-1`.

Any final semantic interpretation of these sentinel values must remain documented separately from the statistical preprocessing logic.

### Numerical features

Continuous and discrete numerical predictors will remain numeric.

Scaling statistics must be learned from the training partition only.

For the initial neural-network pipeline, numerical features that are not binary indicators will be standardized using training-set statistics.

### Missing values

`Base.csv` contains no standard Pandas `NaN` values.

No generic imputation will be introduced unless later validation reveals a real need.

### Derived risk score

`credit_risk_score` remains provisionally included.

Its prediction-time availability must remain documented as a semantic review item before final modeling conclusions are made.

## Leakage Rules

1. Split the data before fitting any preprocessing object.
2. Fit encoders and scalers on training data only.
3. Use `transform`, never `fit_transform`, on validation and test data.
4. Do not calculate preprocessing statistics from the full dataset.
5. Do not use the final test partition to choose preprocessing decisions.
6. Preserve the exact same feature order across train, validation, test, and inference.
7. Save preprocessing metadata so the serving pipeline can reproduce training transformations.

## Milestone 03 Deliverables

By the end of this milestone, the project should contain:

- a reproducible feature manifest
- a train-fitted preprocessing pipeline
- deterministic train/validation/test transformation logic
- validation checks for leakage and feature consistency
- PyTorch-ready feature and target arrays/tensors
- reusable preprocessing code under `src/`
- automated tests
- lightweight preprocessing and split reports

No neural-network training will occur in this milestone.

### Load the dataset and apply the locked split

In [3]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Base.csv"
SPLIT_CONFIG_PATH = PROJECT_ROOT / "configs" / "split_config.json"

# Make the project root importable from the notebook.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.splitting import assign_split, load_split_config

print("Project root:", PROJECT_ROOT)
print("Raw data exists:", RAW_DATA_PATH.exists())
print("Split config exists:", SPLIT_CONFIG_PATH.exists())

split_config = load_split_config(SPLIT_CONFIG_PATH)

df = pd.read_csv(RAW_DATA_PATH)

split_labels = assign_split(df, config=split_config)

print("\nDataset shape:", df.shape)

print("\nSplit counts:")
print(split_labels.value_counts().loc[
    ["train", "validation", "test"]
])

print("\nConfigured test sealed:",
      split_config["test"]["sealed"])

Project root: c:\Users\vrber\ML_Projects\bank-account-fraud-dl-mlops
Raw data exists: True
Split config exists: True

Dataset shape: (1000000, 32)

Split counts:
split
train         794989
validation    108168
test           96843
Name: count, dtype: int64

Configured test sealed: True


### Create separate train, validation, and test DataFrames

In [4]:
train_df = df.loc[split_labels == "train"].copy()
validation_df = df.loc[split_labels == "validation"].copy()
test_df = df.loc[split_labels == "test"].copy()

print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain months:", sorted(train_df["month"].unique().tolist()))
print("Validation months:", sorted(validation_df["month"].unique().tolist()))
print("Test months:", sorted(test_df["month"].unique().tolist()))

print("\nTrain fraud rate:", train_df["fraud_bool"].mean())
print("Validation fraud rate:", validation_df["fraud_bool"].mean())
print("Test fraud rate:", test_df["fraud_bool"].mean())

Train shape: (794989, 32)
Validation shape: (108168, 32)
Test shape: (96843, 32)

Train months: [0, 1, 2, 3, 4, 5]
Validation months: [6]
Test months: [7]

Train fraud rate: 0.010252972053701372
Validation fraud rate: 0.013405073589231566
Test fraud rate: 0.014745515938167962


### Define and validate the feature groups

In [5]:
TARGET_COLUMN = "fraud_bool"

EXCLUDED_FEATURES = [
    "fraud_bool",
    "month",
    "device_fraud_count",
]

CATEGORICAL_FEATURES = [
    "payment_type",
    "employment_status",
    "housing_status",
    "source",
    "device_os",
]

BINARY_FEATURES = [
    "email_is_free",
    "phone_home_valid",
    "phone_mobile_valid",
    "has_other_cards",
    "foreign_request",
    "keep_alive_session",
]

SENTINEL_FEATURES = [
    "prev_address_months_count",
    "current_address_months_count",
    "bank_months_count",
    "session_length_in_minutes",
    "device_distinct_emails_8w",
]

MODEL_FEATURES = [
    col
    for col in df.columns
    if col not in EXCLUDED_FEATURES
]

NUMERIC_FEATURES = [
    col
    for col in MODEL_FEATURES
    if col not in CATEGORICAL_FEATURES
    and col not in BINARY_FEATURES
]

print("Model features:", len(MODEL_FEATURES))
print("Categorical features:", len(CATEGORICAL_FEATURES))
print("Binary features:", len(BINARY_FEATURES))
print("Numeric features:", len(NUMERIC_FEATURES))
print("Sentinel features:", len(SENTINEL_FEATURES))

classified_features = (
    set(CATEGORICAL_FEATURES)
    | set(BINARY_FEATURES)
    | set(NUMERIC_FEATURES)
)

print("\nAll model features classified:",
      classified_features == set(MODEL_FEATURES))

print("Excluded features:")
print(EXCLUDED_FEATURES)

print("\nNumeric features:")
print(NUMERIC_FEATURES)

Model features: 29
Categorical features: 5
Binary features: 6
Numeric features: 18
Sentinel features: 5

All model features classified: True
Excluded features:
['fraud_bool', 'month', 'device_fraud_count']

Numeric features:
['income', 'name_email_similarity', 'prev_address_months_count', 'current_address_months_count', 'customer_age', 'days_since_request', 'intended_balcon_amount', 'zip_count_4w', 'velocity_6h', 'velocity_24h', 'velocity_4w', 'bank_branch_count_8w', 'date_of_birth_distinct_emails_4w', 'credit_risk_score', 'bank_months_count', 'proposed_credit_limit', 'session_length_in_minutes', 'device_distinct_emails_8w']


### Check for unseen categorical values across splits

In [6]:
category_coverage_rows = []

for col in CATEGORICAL_FEATURES:
    train_categories = set(train_df[col].dropna().unique())
    validation_categories = set(validation_df[col].dropna().unique())
    test_categories = set(test_df[col].dropna().unique())

    unseen_in_validation = sorted(
        validation_categories - train_categories
    )

    unseen_in_test = sorted(
        test_categories - train_categories
    )

    category_coverage_rows.append({
        "column": col,
        "train_category_count": len(train_categories),
        "validation_category_count": len(validation_categories),
        "test_category_count": len(test_categories),
        "unseen_in_validation": unseen_in_validation,
        "unseen_in_test": unseen_in_test,
    })

category_coverage = pd.DataFrame(category_coverage_rows)

display(category_coverage)

,column,train_category_count,validation_category_count,test_category_count,unseen_in_validation,unseen_in_test
0,payment_type,5,5,5,[],[]
1,employment_status,7,7,7,[],[]
2,housing_status,7,7,7,[],[]
3,source,2,2,2,[],[]
4,device_os,5,5,5,[],[]


### Add and verify sentinel indicators

In [7]:
def add_sentinel_indicators(dataframe):
    """
    Add binary indicators for features where -1 may represent
    a special unknown/unavailable state.

    The original numeric feature is preserved.
    """
    result = dataframe.copy()

    for col in SENTINEL_FEATURES:
        indicator_col = f"{col}__is_minus_one"

        result[indicator_col] = (
            result[col] == -1
        ).astype("int8")

    return result


train_features = add_sentinel_indicators(
    train_df[MODEL_FEATURES]
)

validation_features = add_sentinel_indicators(
    validation_df[MODEL_FEATURES]
)

test_features = add_sentinel_indicators(
    test_df[MODEL_FEATURES]
)

SENTINEL_INDICATOR_FEATURES = [
    f"{col}__is_minus_one"
    for col in SENTINEL_FEATURES
]

print("Original model features:", len(MODEL_FEATURES))
print("Sentinel indicators added:", len(SENTINEL_INDICATOR_FEATURES))
print("Features before encoding:", train_features.shape[1])

print("\nSentinel indicator columns:")
print(SENTINEL_INDICATOR_FEATURES)

print("\nTrain sentinel counts:")
print(
    train_features[SENTINEL_INDICATOR_FEATURES]
    .sum()
)

Original model features: 29
Sentinel indicators added: 5
Features before encoding: 34

Sentinel indicator columns:
['prev_address_months_count__is_minus_one', 'current_address_months_count__is_minus_one', 'bank_months_count__is_minus_one', 'session_length_in_minutes__is_minus_one', 'device_distinct_emails_8w__is_minus_one']

Train sentinel counts:
prev_address_months_count__is_minus_one       568423
current_address_months_count__is_minus_one      3027
bank_months_count__is_minus_one               199549
session_length_in_minutes__is_minus_one         1513
device_distinct_emails_8w__is_minus_one          309
dtype: int64


### Define the exact preprocessing groups

In [8]:
SCALED_NUMERIC_FEATURES = NUMERIC_FEATURES.copy()

PASSTHROUGH_FEATURES = (
    BINARY_FEATURES
    + SENTINEL_INDICATOR_FEATURES
)

preprocessing_groups = {
    "scaled_numeric": SCALED_NUMERIC_FEATURES,
    "categorical": CATEGORICAL_FEATURES,
    "passthrough": PASSTHROUGH_FEATURES,
}

all_preprocessing_inputs = (
    SCALED_NUMERIC_FEATURES
    + CATEGORICAL_FEATURES
    + PASSTHROUGH_FEATURES
)

print("Scaled numeric:", len(SCALED_NUMERIC_FEATURES))
print("Categorical:", len(CATEGORICAL_FEATURES))
print("Passthrough:", len(PASSTHROUGH_FEATURES))
print("Total preprocessing inputs:", len(all_preprocessing_inputs))

print(
    "\nAll 34 features covered:",
    set(all_preprocessing_inputs)
    == set(train_features.columns)
)

print(
    "No duplicated feature assignments:",
    len(all_preprocessing_inputs)
    == len(set(all_preprocessing_inputs))
)

print("\nPassthrough features:")
print(PASSTHROUGH_FEATURES)

Scaled numeric: 18
Categorical: 5
Passthrough: 11
Total preprocessing inputs: 34

All 34 features covered: True
No duplicated feature assignments: True

Passthrough features:
['email_is_free', 'phone_home_valid', 'phone_mobile_valid', 'has_other_cards', 'foreign_request', 'keep_alive_session', 'prev_address_months_count__is_minus_one', 'current_address_months_count__is_minus_one', 'bank_months_count__is_minus_one', 'session_length_in_minutes__is_minus_one', 'device_distinct_emails_8w__is_minus_one']


### Build the train-fitted preprocessing pipeline

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            SCALED_NUMERIC_FEATURES,
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
            CATEGORICAL_FEATURES,
        ),
        (
            "passthrough",
            "passthrough",
            PASSTHROUGH_FEATURES,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)

print("Preprocessor created.")
print("Preprocessor is fitted:", hasattr(preprocessor, "transformers_"))

Preprocessor created.
Preprocessor is fitted: False


### Fit preprocessing on training data only

In [10]:
preprocessor.fit(train_features)

print(
    "Preprocessor is fitted:",
    hasattr(preprocessor, "transformers_")
)

print(
    "Rows used for fitting:",
    len(train_features)
)

print(
    "Expected training rows:",
    len(train_df)
)

assert len(train_features) == 794989

print("\nTrain-only preprocessing fit: PASS")

Preprocessor is fitted: True
Rows used for fitting: 794989
Expected training rows: 794989

Train-only preprocessing fit: PASS


### Inspect the learned output feature schema

In [11]:
processed_feature_names = (
    preprocessor.get_feature_names_out()
)

print("Processed feature count:", len(processed_feature_names))

print("\nFirst 15 processed features:")
for feature in processed_feature_names[:15]:
    print(feature)

print("\nLast 15 processed features:")
for feature in processed_feature_names[-15:]:
    print(feature)

print("\nLearned categorical vocabularies:")

categorical_encoder = (
    preprocessor
    .named_transformers_["categorical"]
)

for col, categories in zip(
    CATEGORICAL_FEATURES,
    categorical_encoder.categories_,
):
    print(
        f"{col}: {len(categories)} categories -> "
        f"{categories.tolist()}"
    )

Processed feature count: 55

First 15 processed features:
numeric__income
numeric__name_email_similarity
numeric__prev_address_months_count
numeric__current_address_months_count
numeric__customer_age
numeric__days_since_request
numeric__intended_balcon_amount
numeric__zip_count_4w
numeric__velocity_6h
numeric__velocity_24h
numeric__velocity_4w
numeric__bank_branch_count_8w
numeric__date_of_birth_distinct_emails_4w
numeric__credit_risk_score
numeric__bank_months_count

Last 15 processed features:
categorical__device_os_macintosh
categorical__device_os_other
categorical__device_os_windows
categorical__device_os_x11
passthrough__email_is_free
passthrough__phone_home_valid
passthrough__phone_mobile_valid
passthrough__has_other_cards
passthrough__foreign_request
passthrough__keep_alive_session
passthrough__prev_address_months_count__is_minus_one
passthrough__current_address_months_count__is_minus_one
passthrough__bank_months_count__is_minus_one
passthrough__session_length_in_minutes__is_min

### Perform a transform safety check

In [12]:
train_sample = train_features.head(1000)
validation_sample = validation_features.head(1000)
test_sample = test_features.head(1000)

X_train_sample = preprocessor.transform(train_sample)
X_validation_sample = preprocessor.transform(validation_sample)
X_test_sample = preprocessor.transform(test_sample)

print("Train sample shape:", X_train_sample.shape)
print("Validation sample shape:", X_validation_sample.shape)
print("Test sample shape:", X_test_sample.shape)

print("\nExpected feature count:", len(processed_feature_names))

print(
    "\nTrain finite values:",
    np.isfinite(X_train_sample).all()
)

print(
    "Validation finite values:",
    np.isfinite(X_validation_sample).all()
)

print(
    "Test finite values:",
    np.isfinite(X_test_sample).all()
)

assert X_train_sample.shape[1] == 55
assert X_validation_sample.shape[1] == 55
assert X_test_sample.shape[1] == 55

assert np.isfinite(X_train_sample).all()
assert np.isfinite(X_validation_sample).all()
assert np.isfinite(X_test_sample).all()

print("\nSample transformation checks: PASS")

Train sample shape: (1000, 55)
Validation sample shape: (1000, 55)
Test sample shape: (1000, 55)

Expected feature count: 55

Train finite values: True
Validation finite values: True
Test finite values: True

Sample transformation checks: PASS


### Create and verify the target vectors

In [13]:
y_train = train_df[TARGET_COLUMN].to_numpy(dtype=np.float32)
y_validation = validation_df[TARGET_COLUMN].to_numpy(dtype=np.float32)
y_test = test_df[TARGET_COLUMN].to_numpy(dtype=np.float32)

print("y_train shape:", y_train.shape)
print("y_validation shape:", y_validation.shape)
print("y_test shape:", y_test.shape)

print("\nTrain fraud cases:", int(y_train.sum()))
print("Validation fraud cases:", int(y_validation.sum()))
print("Test fraud cases:", int(y_test.sum()))

assert y_train.shape == (794989,)
assert y_validation.shape == (108168,)
assert y_test.shape == (96843,)

assert int(y_train.sum()) == 8151
assert int(y_validation.sum()) == 1450
assert int(y_test.sum()) == 1428

print("\nTarget vector checks: PASS")

y_train shape: (794989,)
y_validation shape: (108168,)
y_test shape: (96843,)

Train fraud cases: 8151
Validation fraud cases: 1450
Test fraud cases: 1428

Target vector checks: PASS


### Estimate transformed matrix memory

In [14]:
def estimate_dense_memory_mb(rows, columns, bytes_per_value):
    return rows * columns * bytes_per_value / (1024 ** 2)


split_sizes = {
    "train": len(train_features),
    "validation": len(validation_features),
    "test": len(test_features),
}

for split_name, rows in split_sizes.items():
    float64_mb = estimate_dense_memory_mb(
        rows,
        len(processed_feature_names),
        8,
    )

    float32_mb = estimate_dense_memory_mb(
        rows,
        len(processed_feature_names),
        4,
    )

    print(
        f"{split_name:10s} "
        f"float64 ≈ {float64_mb:8.2f} MB | "
        f"float32 ≈ {float32_mb:8.2f} MB"
    )

train      float64 ≈   333.59 MB | float32 ≈   166.80 MB
validation float64 ≈    45.39 MB | float32 ≈    22.69 MB
test       float64 ≈    40.64 MB | float32 ≈    20.32 MB


### Verify the current transformed dtype

In [15]:
print("Train sample dtype:", X_train_sample.dtype)
print("Validation sample dtype:", X_validation_sample.dtype)
print("Test sample dtype:", X_test_sample.dtype)

print(
    "\nTrain sample memory:",
    X_train_sample.nbytes / (1024 ** 2),
    "MB",
)

Train sample dtype: float64
Validation sample dtype: float64
Test sample dtype: float64

Train sample memory: 0.41961669921875 MB


### Convert preprocessing inputs to float32

In [16]:
FLOAT32_FEATURES = (
    SCALED_NUMERIC_FEATURES
    + PASSTHROUGH_FEATURES
)

for feature_df in [
    train_features,
    validation_features,
    test_features,
]:
    for col in FLOAT32_FEATURES:
        feature_df[col] = feature_df[col].astype(np.float32)

print("Train numeric dtypes:")
print(
    train_features[SCALED_NUMERIC_FEATURES]
    .dtypes
    .value_counts()
)

print("\nTrain passthrough dtypes:")
print(
    train_features[PASSTHROUGH_FEATURES]
    .dtypes
    .value_counts()
)

print("\nCategorical dtypes:")
print(
    train_features[CATEGORICAL_FEATURES]
    .dtypes
    .value_counts()
)

Train numeric dtypes:
float32    18
Name: count, dtype: int64

Train passthrough dtypes:
float32    11
Name: count, dtype: int64

Categorical dtypes:
str    5
Name: count, dtype: int64


### Rebuild the preprocessor for float32 output

In [17]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            SCALED_NUMERIC_FEATURES,
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
                dtype=np.float32,
            ),
            CATEGORICAL_FEATURES,
        ),
        (
            "passthrough",
            "passthrough",
            PASSTHROUGH_FEATURES,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)

# Fit ONLY on training data
preprocessor.fit(train_features)

processed_feature_names = (
    preprocessor.get_feature_names_out()
)

# Small verification transform
X_train_sample_float32 = preprocessor.transform(
    train_features.head(1000)
)

print("Preprocessor fitted:", hasattr(preprocessor, "transformers_"))
print("Processed feature count:", len(processed_feature_names))
print("Sample shape:", X_train_sample_float32.shape)
print("Sample dtype:", X_train_sample_float32.dtype)

print(
    "Sample memory:",
    X_train_sample_float32.nbytes / (1024 ** 2),
    "MB",
)

assert X_train_sample_float32.shape == (1000, 55)
assert X_train_sample_float32.dtype == np.float32
assert np.isfinite(X_train_sample_float32).all()

print("\nFloat32 preprocessing check: PASS")

Preprocessor fitted: True
Processed feature count: 55
Sample shape: (1000, 55)
Sample dtype: float32
Sample memory: 0.209808349609375 MB

Float32 preprocessing check: PASS


### Transform the full training set

In [18]:
X_train = preprocessor.transform(train_features)

print("X_train shape:", X_train.shape)
print("X_train dtype:", X_train.dtype)

print(
    "X_train memory:",
    round(X_train.nbytes / (1024 ** 2), 2),
    "MB",
)

assert X_train.shape == (
    len(train_features),
    len(processed_feature_names),
)

assert X_train.dtype == np.float32

print("\nFull training transformation: PASS")

X_train shape: (794989, 55)
X_train dtype: float32
X_train memory: 166.8 MB

Full training transformation: PASS


### Transform the full validation set

In [19]:
X_validation = preprocessor.transform(
    validation_features
)

print("X_validation shape:", X_validation.shape)
print("X_validation dtype:", X_validation.dtype)

print(
    "X_validation memory:",
    round(
        X_validation.nbytes / (1024 ** 2),
        2,
    ),
    "MB",
)

assert X_validation.shape == (
    len(validation_features),
    len(processed_feature_names),
)

assert X_validation.dtype == np.float32
assert np.isfinite(X_validation).all()

print("\nFull validation transformation: PASS")

X_validation shape: (108168, 55)
X_validation dtype: float32
X_validation memory: 22.69 MB

Full validation transformation: PASS


### Run a development-matrix integrity check

In [20]:
print("Checking training matrix...")
print("Finite values:", np.isfinite(X_train).all())
print("Rows align with y:", X_train.shape[0] == y_train.shape[0])

print("\nChecking validation matrix...")
print("Finite values:", np.isfinite(X_validation).all())
print(
    "Rows align with y:",
    X_validation.shape[0] == y_validation.shape[0],
)

print("\nChecking feature schema...")
print(
    "Feature names unique:",
    len(processed_feature_names)
    == len(set(processed_feature_names)),
)

for excluded_col in [
    "fraud_bool",
    "month",
    "device_fraud_count",
]:
    leaked = [
        feature
        for feature in processed_feature_names
        if feature.endswith(f"__{excluded_col}")
    ]

    print(
        f"{excluded_col} present in processed features:",
        bool(leaked),
    )

assert np.isfinite(X_train).all()
assert np.isfinite(X_validation).all()

assert X_train.shape[0] == y_train.shape[0]
assert X_validation.shape[0] == y_validation.shape[0]

assert X_train.shape[1] == 55
assert X_validation.shape[1] == 55

assert len(processed_feature_names) == 55
assert len(set(processed_feature_names)) == 55

assert not any(
    feature.endswith("__fraud_bool")
    for feature in processed_feature_names
)

assert not any(
    feature.endswith("__month")
    for feature in processed_feature_names
)

assert not any(
    feature.endswith("__device_fraud_count")
    for feature in processed_feature_names
)

print("\nDevelopment matrix integrity checks: PASS")

Checking training matrix...
Finite values: True
Rows align with y: True

Checking validation matrix...
Finite values: True
Rows align with y: True

Checking feature schema...
Feature names unique: True
fraud_bool present in processed features: False
month present in processed features: False
device_fraud_count present in processed features: False

Development matrix integrity checks: PASS


### Check whether PyTorch is already installed

In [21]:
import importlib.util

torch_available = importlib.util.find_spec("torch") is not None

print("PyTorch installed:", torch_available)

if torch_available:
    import torch

    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

PyTorch installed: True
PyTorch version: 2.14.1+cpu
CUDA available: False


### Verify PyTorch

In [22]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

test_tensor = torch.tensor(
    [[1.0, 2.0], [3.0, 4.0]],
    dtype=torch.float32,
)

print("Test tensor:")
print(test_tensor)
print("Tensor dtype:", test_tensor.dtype)

assert torch.__version__.endswith("+cpu")
assert test_tensor.dtype == torch.float32

print("\nPyTorch installation check: PASS")

PyTorch version: 2.14.1+cpu
CUDA available: False
Test tensor:
tensor([[1., 2.],
        [3., 4.]])
Tensor dtype: torch.float32

PyTorch installation check: PASS


In [23]:
import torch
print(torch.__version__)

2.14.1+cpu


### Wrap the development arrays as PyTorch datasets

In [24]:
import torch
from torch.utils.data import TensorDataset

X_train_tensor = torch.from_numpy(X_train)
y_train_tensor = torch.from_numpy(y_train)

X_validation_tensor = torch.from_numpy(X_validation)
y_validation_tensor = torch.from_numpy(y_validation)

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor,
)

validation_dataset = TensorDataset(
    X_validation_tensor,
    y_validation_tensor,
)

print("Train tensor shape:", X_train_tensor.shape)
print("Train target shape:", y_train_tensor.shape)
print("Validation tensor shape:", X_validation_tensor.shape)
print("Validation target shape:", y_validation_tensor.shape)

print("\nFeature dtype:", X_train_tensor.dtype)
print("Target dtype:", y_train_tensor.dtype)

print("\nTrain dataset length:", len(train_dataset))
print("Validation dataset length:", len(validation_dataset))

assert X_train_tensor.dtype == torch.float32
assert y_train_tensor.dtype == torch.float32
assert X_validation_tensor.dtype == torch.float32
assert y_validation_tensor.dtype == torch.float32

assert len(train_dataset) == 794989
assert len(validation_dataset) == 108168

print("\nPyTorch dataset checks: PASS")

Train tensor shape: torch.Size([794989, 55])
Train target shape: torch.Size([794989])
Validation tensor shape: torch.Size([108168, 55])
Validation target shape: torch.Size([108168])

Feature dtype: torch.float32
Target dtype: torch.float32

Train dataset length: 794989
Validation dataset length: 108168

PyTorch dataset checks: PASS


### Create the PyTorch DataLoaders

In [25]:
from torch.utils.data import DataLoader

BATCH_SIZE = 1024
RANDOM_SEED = 42

generator = torch.Generator()
generator.manual_seed(RANDOM_SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    generator=generator,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

X_batch, y_batch = next(iter(train_loader))

print("Batch size:", BATCH_SIZE)
print("Train batches:", len(train_loader))
print("Validation batches:", len(validation_loader))

print("\nFirst training batch:")
print("X shape:", X_batch.shape)
print("y shape:", y_batch.shape)
print("X dtype:", X_batch.dtype)
print("y dtype:", y_batch.dtype)

assert X_batch.shape[1] == 55
assert X_batch.dtype == torch.float32
assert y_batch.dtype == torch.float32

print("\nPyTorch DataLoader checks: PASS")

Batch size: 1024
Train batches: 777
Validation batches: 106

First training batch:
X shape: torch.Size([1024, 55])
y shape: torch.Size([1024])
X dtype: torch.float32
y dtype: torch.float32

PyTorch DataLoader checks: PASS


### Generate the Milestone 03 feature manifest

In [26]:
category_widths = {
    column: len(categories)
    for column, categories in zip(
        CATEGORICAL_FEATURES,
        preprocessor
        .named_transformers_["categorical"]
        .categories_,
    )
}

manifest_rows = []

for column in SCALED_NUMERIC_FEATURES:
    manifest_rows.append({
        "feature": column,
        "source_feature": column,
        "feature_group": "numeric",
        "derived": False,
        "preprocessing": "StandardScaler",
        "fit_scope": "train_only",
        "output_width": 1,
        "output_dtype": "float32",
    })

for column in CATEGORICAL_FEATURES:
    manifest_rows.append({
        "feature": column,
        "source_feature": column,
        "feature_group": "categorical",
        "derived": False,
        "preprocessing": "OneHotEncoder(handle_unknown='ignore')",
        "fit_scope": "train_only",
        "output_width": category_widths[column],
        "output_dtype": "float32",
    })

for column in BINARY_FEATURES:
    manifest_rows.append({
        "feature": column,
        "source_feature": column,
        "feature_group": "binary",
        "derived": False,
        "preprocessing": "passthrough",
        "fit_scope": "none",
        "output_width": 1,
        "output_dtype": "float32",
    })

for source_column, indicator_column in zip(
    SENTINEL_FEATURES,
    SENTINEL_INDICATOR_FEATURES,
):
    manifest_rows.append({
        "feature": indicator_column,
        "source_feature": source_column,
        "feature_group": "sentinel_indicator",
        "derived": True,
        "preprocessing": "indicator(value == -1) + passthrough",
        "fit_scope": "none",
        "output_width": 1,
        "output_dtype": "float32",
    })

feature_manifest = pd.DataFrame(manifest_rows)

manifest_path = (
    PROJECT_ROOT
    / "reports"
    / "milestone_03_feature_manifest.csv"
)

feature_manifest.to_csv(
    manifest_path,
    index=False,
)

print("Manifest path:", manifest_path)
print("Manifest rows:", len(feature_manifest))
print(
    "Final transformed width:",
    feature_manifest["output_width"].sum(),
)

display(feature_manifest)

Manifest path: c:\Users\vrber\ML_Projects\bank-account-fraud-dl-mlops\reports\milestone_03_feature_manifest.csv
Manifest rows: 34
Final transformed width: 55


,feature,source_feature,feature_group,derived,preprocessing,fit_scope,output_width,output_dtype
0,income,income,numeric,False,StandardScaler,train_only,1,float32
1,name_email_similarity,name_email_similarity,numeric,False,StandardScaler,train_only,1,float32
2,prev_address_months_count,prev_address_months_count,numeric,False,StandardScaler,train_only,1,float32
3,current_address_months_count,current_address_months_count,numeric,False,StandardScaler,train_only,1,float32
4,customer_age,customer_age,numeric,False,StandardScaler,train_only,1,float32
5,days_since_request,days_since_request,numeric,False,StandardScaler,train_only,1,float32
6,intended_balcon_amount,intended_balcon_amount,numeric,False,StandardScaler,train_only,1,float32
7,zip_count_4w,zip_count_4w,numeric,False,StandardScaler,train_only,1,float32
8,velocity_6h,velocity_6h,numeric,False,StandardScaler,train_only,1,float32
9,velocity_24h,velocity_24h,numeric,False,StandardScaler,train_only,1,float32


### Create the Milestone 03 preprocessing metrics report

In [27]:
preprocessing_metrics = pd.DataFrame([
    {
        "metric": "raw_model_features",
        "value": len(MODEL_FEATURES),
    },
    {
        "metric": "pre_encoding_features",
        "value": train_features.shape[1],
    },
    {
        "metric": "final_transformed_features",
        "value": len(processed_feature_names),
    },
    {
        "metric": "numeric_features",
        "value": len(SCALED_NUMERIC_FEATURES),
    },
    {
        "metric": "categorical_features",
        "value": len(CATEGORICAL_FEATURES),
    },
    {
        "metric": "binary_features",
        "value": len(BINARY_FEATURES),
    },
    {
        "metric": "sentinel_indicators",
        "value": len(SENTINEL_INDICATOR_FEATURES),
    },
    {
        "metric": "train_rows",
        "value": X_train.shape[0],
    },
    {
        "metric": "validation_rows",
        "value": X_validation.shape[0],
    },
    {
        "metric": "train_batches_batch_size_1024",
        "value": len(train_loader),
    },
    {
        "metric": "validation_batches_batch_size_1024",
        "value": len(validation_loader),
    },
    {
        "metric": "output_dtype",
        "value": str(X_train.dtype),
    },
    {
        "metric": "test_full_matrix_materialized",
        "value": False,
    },
])

metrics_path = (
    PROJECT_ROOT
    / "reports"
    / "milestone_03_preprocessing_metrics.csv"
)

preprocessing_metrics.to_csv(
    metrics_path,
    index=False,
)

print("Metrics path:", metrics_path)
print("Metrics rows:", len(preprocessing_metrics))

display(preprocessing_metrics)

Metrics path: c:\Users\vrber\ML_Projects\bank-account-fraud-dl-mlops\reports\milestone_03_preprocessing_metrics.csv
Metrics rows: 13


,metric,value
0,raw_model_features,29
1,pre_encoding_features,34
2,final_transformed_features,55
3,numeric_features,18
4,categorical_features,5
5,binary_features,6
6,sentinel_indicators,5
7,train_rows,794989
8,validation_rows,108168
9,train_batches_batch_size_1024,777


### Save the column transformed feature order

In [28]:
feature_order = pd.DataFrame({
    "position": range(len(processed_feature_names)),
    "processed_feature": processed_feature_names,
})

feature_order_path = (
    PROJECT_ROOT
    / "reports"
    / "milestone_03_processed_feature_order.csv"
)

feature_order.to_csv(
    feature_order_path,
    index=False,
)

print("Feature order path:", feature_order_path)
print("Feature count:", len(feature_order))
print("Unique feature names:", feature_order["processed_feature"].is_unique)

print("\nFirst 5:")
display(feature_order.head())

print("\nLast 5:")
display(feature_order.tail())

Feature order path: c:\Users\vrber\ML_Projects\bank-account-fraud-dl-mlops\reports\milestone_03_processed_feature_order.csv
Feature count: 55
Unique feature names: True

First 5:


,position,processed_feature
0,0,numeric__income
1,1,numeric__name_email_similarity
2,2,numeric__prev_address_months_count
3,3,numeric__current_address_months_count
4,4,numeric__customer_age



Last 5:


,position,processed_feature
50,50,passthrough__prev_address_months_count__is_min...
51,51,passthrough__current_address_months_count__is_...
52,52,passthrough__bank_months_count__is_minus_one
53,53,passthrough__session_length_in_minutes__is_min...
54,54,passthrough__device_distinct_emails_8w__is_min...


### Verify reusable code matches the notebook pipeline

In [29]:
from src.features.preprocessing import (
    build_preprocessor as build_reusable_preprocessor,
    load_preprocessing_config as load_reusable_config,
    prepare_model_features as prepare_reusable_features,
)

reusable_config = load_reusable_config()

reusable_train_features = prepare_reusable_features(
    train_df,
    reusable_config,
)

reusable_validation_features = prepare_reusable_features(
    validation_df,
    reusable_config,
)

reusable_preprocessor = build_reusable_preprocessor(
    reusable_config
)

# Fit ONLY on training data
reusable_preprocessor.fit(
    reusable_train_features
)

reusable_feature_names = (
    reusable_preprocessor.get_feature_names_out()
)

# Compare using small samples to avoid unnecessary memory duplication
reusable_train_sample = reusable_preprocessor.transform(
    reusable_train_features.head(1000)
)

reusable_validation_sample = reusable_preprocessor.transform(
    reusable_validation_features.head(1000)
)

notebook_train_sample = preprocessor.transform(
    train_features.head(1000)
)

notebook_validation_sample = preprocessor.transform(
    validation_features.head(1000)
)

print(
    "Feature names identical:",
    np.array_equal(
        reusable_feature_names,
        processed_feature_names,
    ),
)

print(
    "Train sample transformations identical:",
    np.allclose(
        reusable_train_sample,
        notebook_train_sample,
    ),
)

print(
    "Validation sample transformations identical:",
    np.allclose(
        reusable_validation_sample,
        notebook_validation_sample,
    ),
)

print(
    "Reusable output dtype:",
    reusable_train_sample.dtype,
)

assert np.array_equal(
    reusable_feature_names,
    processed_feature_names,
)

assert np.allclose(
    reusable_train_sample,
    notebook_train_sample,
)

assert np.allclose(
    reusable_validation_sample,
    notebook_validation_sample,
)

assert reusable_train_sample.dtype == np.float32

print("\nNotebook ↔ reusable preprocessing equivalence: PASS")

Feature names identical: True
Train sample transformations identical: True
Validation sample transformations identical: True
Reusable output dtype: float32

Notebook ↔ reusable preprocessing equivalence: PASS


## Milestone 03 Interpretation and Takeaways

Milestone 03 converted the audited fraud dataset into a leakage-safe feature pipeline that can be reused by both classical machine-learning baselines and the later PyTorch neural network.

### What was established

- The chronological split from Milestone 02 remains unchanged:
  - Train: months 0–5
  - Validation: month 6
  - Test: month 7
- Learned preprocessing is fit on training data only.
- `fraud_bool`, `month`, and `device_fraud_count` are excluded from the initial model feature matrix.
- 18 numeric features are standardized.
- 5 categorical features are one-hot encoded with unknown-category protection.
- 6 existing binary indicators are passed through unchanged.
- 5 sentinel-coded numeric features retain their original value and receive an additional `__is_minus_one` indicator.
- 29 raw model features become 34 pre-encoding features and finally 55 transformed features.
- The transformed matrices use `float32` to reduce memory usage and match PyTorch expectations.
- Training and validation tensors and DataLoaders were successfully constructed.
- The full month-7 test matrix was not materialized and remains sealed for Milestone 07.

### Development matrix

- Train: 794,989 rows × 55 features
- Validation: 108,168 rows × 55 features
- Batch size: 1,024
- Train batches: 777
- Validation batches: 106

### Reusability

The notebook implementation was compared directly against the reusable preprocessing implementation in `src/features/preprocessing.py`.

The feature names, training transformations, and validation transformations were identical.

This confirms that the exploratory notebook and reusable project code have not drifted apart.

### Key lesson

Preprocessing is part of the model system, not merely a cleaning step.

Any transformation that learns information from data—such as scaling statistics or categorical vocabularies—must obey the same train/validation/test boundaries as model training itself. Otherwise, information from later data can leak into model development and make evaluation results overly optimistic.

### Next milestone

Milestone 04 will establish baseline models and the evaluation framework.

The purpose of those baselines will be to create a defensible reference point for the later PyTorch neural network rather than assuming that deep learning is automatically superior.

## Interview-ready explanation

Practice explaining the milestone in about 60–90 seconds:

> “In Milestone 03, I built a leakage-safe preprocessing pipeline for the bank account fraud dataset that can be reused by both classical machine-learning models and the later PyTorch neural network. I preserved the chronological split from Milestone 02, using months 0–5 for training, month 6 for validation, and keeping month 7 sealed for final evaluation. I fit all learned preprocessing only on the training data. I standardized 18 numeric features, one-hot encoded 5 categorical features with unknown-category protection, passed through 6 binary indicators, and created 5 additional sentinel indicators for numeric features where `-1` may represent an unavailable or special state. The 29 raw model features became 55 final transformed features. I converted the matrices to `float32` for lower memory usage and PyTorch compatibility, then verified that the reusable preprocessing module produced exactly the same feature names and transformations as the notebook implementation.”

### Questions you should be ready to answer

1. Why did you fit the preprocessing pipeline only on the training data?
2. Why did you use `OneHotEncoder(handle_unknown="ignore")`?
3. How did you handle the `-1` sentinel values?
4. Why did you use `float32` instead of `float64`?
5. Why did you keep the test set sealed during preprocessing?

## Preprocessing Detailed Answers

### 1. Why did you fit the preprocessing pipeline only on the training data?

I fit all learned preprocessing steps only on the training partition to prevent data leakage. The project uses a chronological split where months 0–5 are training data, month 6 is validation data, and month 7 is the final test data.

Transformations such as `StandardScaler` and `OneHotEncoder` learn information from the data. `StandardScaler` learns statistics such as the mean and standard deviation of each numeric feature, while `OneHotEncoder` learns which categorical values exist and how they should be represented.

If I fit those transformations using validation or test data, information from later time periods would influence the preprocessing used during model development. That would make the evaluation less realistic.

### 2. Why did you use OneHotEncoder(handle_unknown="ignore")?
I used OneHotEncoder(handle_unknown="ignore") because categorical values that appear in future production data may not have existed in the training dataset.
The project has five categorical features:
- payment_type
- employment_status
- housing_status
- source
- device_os
During the current dataset analysis, the validation and test periods did not contain unseen categories. However, that does not guarantee that future production data will behave the same way.
Without handle_unknown="ignore", an unseen category could cause the preprocessing pipeline to fail during inference. With this setting, the pipeline can continue processing the observation without raising an error.
This is an example of designing the preprocessing pipeline for deployment rather than only for the historical dataset.

### 3. How did you handle the -1 sentinel values?
I did not automatically replace every -1 value with a mean, median, or other imputed value because the data audit suggested that -1 may represent a meaningful unavailable or unknown state.
Five numeric features were identified as sentinel candidates:
- prev_address_months_count
- current_address_months_count
- bank_months_count
- session_length_in_minutes
- device_distinct_emails_8w
For each of these features, I kept the original numeric value and created an additional binary indicator showing whether the value was -1.

### 4. Why did you use float32 instead of float64?
Using float64, the training matrix would require approximately 334 MB of memory. Using float32, it requires approximately 167 MB.
That reduces the memory requirement by about half, which matters because this project is being developed on a local machine and later needs to create tensors, DataLoaders, model parameters, gradients, and other training objects in memory.
PyTorch neural networks also commonly operate using torch.float32, so producing the preprocessing output directly as float32 avoids unnecessary conversions and additional memory copies.

### 5. Why did you keep the test set sealed during preprocessing?
I kept the month-7 test partition sealed because it is intended to represent the final unseen period used to estimate how well the completed system generalizes.
The test set should not influence decisions such as:
- feature selection
- preprocessing design
- model selection
- neural-network architecture
- hyperparameter tuning
- class-imbalance strategy
- calibration
- threshold selection
- business-cost optimization

If I repeatedly inspect the test results while making those decisions, I would gradually optimize the project for the test set. At that point, the test set would no longer be truly unseen.
During Milestone 03, I only performed a small compatibility transformation to confirm that the train-fitted preprocessing pipeline could process the expected schema. I did not materialize or analyze the full transformed test matrix.
The full month-7 test evaluation is reserved for Milestone 07, after the preprocessing pipeline, model configuration, calibration strategy, threshold policy, and evaluation procedure have been finalized.
This gives the final test results much more credibility because they represent a genuine holdout period rather than another development dataset.